In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


data_dir_rerun = Path(r"C:\Users\myles\OneDrive\Desktop\data_new")  
files_rerun = sorted(data_dir_rerun.glob(("run_phi*_mus*_mur*")))
print("Rerun files found:", len(files_rerun))

def find_table_start(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Time"):
                return i
    raise ValueError(f"No table header found in {path}")


def find_table_end(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Loop"):
                return i
    return None  # If no end marker is found, return Non

In [ ]:
"""Extracts the relevant stress_xy data column from each data set and combines them into one dataframe.  """

columns_rerun = []
for file in files_rerun:
    start = find_table_start(file)
    end = find_table_end(file)
    df = pd.read_csv(file, delim_whitespace=True, skiprows=start,
                     nrows=end - start if end is not None else None,
                     on_bad_lines="skip", engine="python")
    col = df.iloc[:, 4].rename(f"{Path(file).stem}_stress_xy_rerun")
    columns_rerun.append(col)

df_rerun = pd.concat(columns_rerun, axis=1)
df_rerun.insert(0, "Time", range(len(df_rerun)))

# Caluclates reduced viscosity
eta_f = 0.1; gamma_dot = 0.01
for col in [c for c in df_rerun.columns if c != "Time"]:
    df_rerun[f"reduced_{col}"] = np.abs(df_rerun[col] / (eta_f * gamma_dot))

# Steady-state selected
steady_rerun = df_rerun[(df_rerun["Time"] >= 200)]
reduced_cols_rerun = [c for c in df_rerun.columns if c.startswith("reduced_")]

avg_rerun = steady_rerun[reduced_cols_rerun].mean()
tau_strain = 1.0                          # 1 strain unit, as advised
dt_strain  = gamma_dot                    # each timestep = gamma_dot strain units (= 0.01)
tau_steps  = tau_strain / dt_strain       # = 100 timesteps per correlation time
N          = len(steady_rerun)
n_eff      = N / (2 * tau_steps + 1)     # effective independent samples
std_rerun  = steady_rerun[reduced_cols_rerun].std() / np.sqrt(n_eff)


fric_1_1_rerun = avg_rerun[avg_rerun.index.str.contains("mus1_mur1")]
sem_1_1_rerun  = std_rerun[std_rerun.index.str.contains("mus1_mur1")].values

fric_1_0_rerun = avg_rerun[avg_rerun.index.str.contains("mus1_mur0")]
sem_1_0_rerun  = std_rerun[std_rerun.index.str.contains("mus1_mur0")].values

fric_0_0_rerun = avg_rerun[avg_rerun.index.str.contains("mus0_mur0")]
sem_0_0_rerun  = std_rerun[std_rerun.index.str.contains("mus0_mur0")].values

x_1_1_rerun = [0.35, 0.36, 0.37, 0.38, 0.39, 0.40, 0.41, 0.42, 0.43, 0.44, 0.45]   #  phi values
y_1_1_rerun = np.sort(fric_1_1_rerun.values)

x_1_0_rerun = [0.50, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57]   #  phi values
y_1_0_rerun = np.sort(fric_1_0_rerun.values)

x_0_0_rerun = [0.55, 0.56, 0.57, 0.58, 0.59, 0.60, 0.61, 0.62, 0.63, 0.64]   # phi values
y_0_0_rerun = np.sort(fric_0_0_rerun.values)

In [ ]:
# Power law form used to fit viscosity as a function of packing fraction
def func_fixed(x, xm):
    return (1-x/xm)**(-2)

# Friction 1,1 fit
params_1_1_rerun, _ = curve_fit(func_fixed, x_1_1_rerun, y_1_1_rerun, p0=[0.4505] , maxfev=10000000)
x_1_1_rerun_fit = np.linspace(0.20, params_1_1_rerun[0], 200)
y_1_1_rerun_fit = func_fixed(x_1_1_rerun_fit, *params_1_1_rerun)

# Friction 1,0 fit
params_1_0_rerun, _ = curve_fit(func_fixed, x_1_0_rerun, y_1_0_rerun, p0=[0.5702], maxfev=10000)
x_1_0_rerun_fit = np.linspace(0.20, params_1_0_rerun[0], 200)
y_1_0_rerun_fit = func_fixed(x_1_0_rerun_fit, *params_1_0_rerun)

# Friction 0,0 fit
params_0_0_rerun, _ = curve_fit(func_fixed, x_0_0_rerun, y_0_0_rerun, p0=[0.6477], maxfev=100000)
x_0_0_rerun_fit = np.linspace(0.20, params_0_0_rerun[0], 200)
y_0_0_rerun_fit = func_fixed(x_0_0_rerun_fit, *params_0_0_rerun)

# Plot the three curves together for comparison, (curves run from 0.20 to their xm value)
plt.figure(figsize=(8,6))

plt.errorbar(x_1_1_rerun, y_1_1_rerun, 
             label='muS1, muR1 (rerun)', marker='s', linestyle='none',
             color='darkred', alpha=0.7)

plt.errorbar(x_1_0_rerun, y_1_0_rerun, 
             label='muS1, muR0 (rerun)', marker='s', linestyle='none',
             color='darkblue', alpha=0.7)

plt.errorbar(x_0_0_rerun, y_0_0_rerun, 
             label='muS0, muR0 (rerun)', marker='s', linestyle='none',
             color='dimgray', alpha=0.7)

plt.plot(x_1_1_rerun_fit, y_1_1_rerun_fit, label=f'muS1, muR1 fit', color='darkred')

plt.plot(x_1_0_rerun_fit, y_1_0_rerun_fit, label=f'muS1, muR0 fit', color='darkblue')

plt.plot(x_0_0_rerun_fit, y_0_0_rerun_fit, label=f'muS0, muR0 fit', color='dimgray')

# Add jamming fraction lines
plt.axvline(x=params_1_1_rerun[0], color='red', linestyle='--', label='Fitted Jamming Fraction (muS1, muR1)')
plt.axvline(x=params_1_0_rerun[0], color='blue', linestyle='--', label='Fitted Jamming Fraction (muS1, muR0)')
plt.axvline(x=params_0_0_rerun[0], color='black', linestyle='--', label='Fitted Jamming Fraction (muS0, muR0)')

plt.xlabel("$\phi$", fontsize=18, fontweight='bold')
plt.minorticks_on()
plt.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.ylabel('$\eta_r$', fontsize=18, fontweight='bold')
plt.xticks(fontsize=0)
plt.yticks(fontsize=12)

plt.ylim(1, 1e5)
plt.yscale('log')
plt.xlim(0.20, 0.70)
plt.grid(False)
plt.show()

print("Fitted parameters for muS1, muR1 (xm, A, B):", params_1_1_rerun)
print("Fitted parameters for muS1, muR0 (xm, A, B):", params_1_0_rerun)
print("Fitted parameters for muS0, muR0 (xm, A, B):", params_0_0_rerun)

In [ ]:
"""Following similar procedure as cell 2, coordination data extracted from column 8 in each data file, and formed into dataframe"""

columns_rerun_z = []  

for file in files_rerun:
    start = find_table_start(file)
    end = find_table_end(file)
    df = pd.read_csv(
        file,
        sep=r"\s+",
        skiprows=start,
        nrows=end - start if end is not None else None,
        on_bad_lines="skip",
        engine="python"
    )
    col_1 = df.iloc[:, 7].rename(f"{Path(file).stem}_stress_xy")
    columns_rerun_z.append(col_1)

print(f"Collected {len(columns_rerun_z)} columns")  

contact_df = pd.concat(columns_rerun_z, axis=1)
contact_df.insert(0, "Time", range(len(contact_df)))
contact_df.head()



In [ ]:
# Steady-state coordination number
steady_contact_df = contact_df[(contact_df["Time"] >= 200)]
reduced_cols = [col for col in steady_contact_df.columns if col != "Time"]
print("Reduced columns:", reduced_cols)

# Average steady-state value
avg_contact = steady_contact_df[reduced_cols].mean()
print("Average Coordinate Number in Steady State:")
print(avg_contact)

fric_1_1_rerun_z = avg_contact[avg_contact.index.str.contains("mus1_mur1")]

fric_1_0_rerun_z = avg_contact[avg_contact.index.str.contains("mus1_mur0")]

fric_0_0_rerun_z = avg_contact[avg_contact.index.str.contains("mus0_mur0")]


y_1_1_rerun_z = np.sort(fric_1_1_rerun_z.values)
y_1_0_rerun_z = np.sort(fric_1_0_rerun_z.values)
y_0_0_rerun_z = np.sort(fric_0_0_rerun_z.values)



In [ ]:
# Plot the three curves together for comparison, (curves run from 0.20 to their xm value)
plt.figure(figsize=(8,6))

plt.errorbar(x_1_1_rerun, y_1_1_rerun_z, 
             label='muS1, muR1 (rerun)', marker='s', linestyle='none',
             color='darkred', alpha=0.7)

plt.errorbar(x_1_0_rerun, y_1_0_rerun_z, 
             label='muS1, muR0 (rerun)', marker='s', linestyle='none',
             color='darkblue', alpha=0.7)

plt.errorbar(x_0_0_rerun, y_0_0_rerun_z, 
             label='muS0, muR0 (rerun)', marker='s', linestyle='none',
             color='dimgray', alpha=0.7)


# Add jamming fraction lines
plt.axvline(x=params_1_1_rerun[0], color='red', linestyle='--', label='Fitted Jamming Fraction (muS1, muR1)')
plt.axvline(x=params_1_0_rerun[0], color='blue', linestyle='--', label='Fitted Jamming Fraction (muS1, muR0)')
plt.axvline(x=params_0_0_rerun[0], color='black', linestyle='--', label='Fitted Jamming Fraction (muS0, muR0)')

# Add isostatic lines
plt.axhline(y=4,   color='b',     linestyle='--', label='Isostatic Coordinate Number = 4')
plt.axhline(y=2.4, color='r',     linestyle='--', label='Fitted Jamming Coordinate Number = 2.4')
plt.axhline(y=6,   color='black', linestyle='--', label='Maximum Coordinate Number = 6')

plt.xlabel("$\phi$", fontsize=18, fontweight='bold')
plt.minorticks_on()
plt.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.ylabel('Z', fontsize=18, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.xlim(0.20, 0.7)
plt.ylim(0,6.5)
plt.grid(False)
plt.show()

print("Fitted parameters for muS1, muR1 (xm, A, B):", params_1_1_rerun)
print("Fitted parameters for muS1, muR0 (xm, A, B):", params_1_0_rerun)
print("Fitted parameters for muS0, muR0 (xm, A, B):", params_0_0_rerun)


In [ ]:
"""Combines the two plots for the report"""

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    figsize=(8, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1], "hspace": 0.05}
)

# === TOP: relative viscosity ===
ax_top.errorbar(x_1_1_rerun, y_1_1_rerun, yerr=sem_1_1_rerun,
                label='muS1, muR1', marker='s', linestyle='none',
                color='darkred', alpha=1)
ax_top.errorbar(x_1_0_rerun, y_1_0_rerun, yerr=sem_1_0_rerun,
                label='muS1, muR0', marker='s', linestyle='none',
                color='darkblue', alpha=1)
ax_top.errorbar(x_0_0_rerun, y_0_0_rerun, yerr=sem_0_0_rerun,
                label='muS0, muR0', marker='s', linestyle='none',
                color='dimgray', alpha=1)
ax_top.plot(x_1_1_rerun_fit, y_1_1_rerun_fit, color='darkred')
ax_top.plot(x_1_0_rerun_fit, y_1_0_rerun_fit, color='darkblue')
ax_top.plot(x_0_0_rerun_fit, y_0_0_rerun_fit, color='dimgray')
ax_top.set_ylabel(r'$\eta_r$', fontsize=18, fontweight='bold')
ax_top.set_yscale('log')
ax_top.set_ylim(1, 1e5)
ax_top.set_xlim(0.20, 0.70)
ax_top.minorticks_on()
ax_top.tick_params(axis='both', which='both', direction='in',
                   top=True, right=True, labelsize=12)
ax_top.grid(False)

# === BOTTOM: coordination number ===
ax_bot.errorbar(x_1_1_rerun, y_1_1_rerun_z,
                label='muS1, muR1', marker='s', linestyle='none',
                color='darkred', alpha=1)
ax_bot.errorbar(x_1_0_rerun, y_1_0_rerun_z,
                label='muS1, muR0', marker='s', linestyle='none',
                color='darkblue', alpha=1)
ax_bot.errorbar(x_0_0_rerun, y_0_0_rerun_z,
                label='muS0, muR0', marker='s', linestyle='none',
                color='dimgray', alpha=1)

# Isostatic lines
ax_bot.axhline(y=4,   color='b',     linestyle=':', label='Isostatic Z = 4')
ax_bot.axhline(y=2.4, color='r',     linestyle=':', label='Jamming Z = 2.4')
ax_bot.axhline(y=6,   color='black', linestyle=':', label='Max Z = 6')

ax_bot.set_xlabel(r'$\phi$', fontsize=18, fontweight='bold')
ax_bot.set_ylabel('Average Coordinate Number, Z', fontsize=12, fontweight='bold')
ax_bot.set_ylim(0, 6.5)
ax_bot.minorticks_on()
ax_bot.tick_params(axis='both', which='both', direction='in',
                   top=True, right=True, labelsize=12)
ax_bot.grid(False)

# Vertical jamming lines across both panels ===
jam_lines = [
    (params_1_1_rerun[0], 'red',   r'Jamming $\phi$ (muS1, muR1)'),
    (params_1_0_rerun[0], 'blue',  r'Jamming $\phi$ (muS1, muR0)'),   # fixed
    (params_0_0_rerun[0], 'black', r'Jamming $\phi$ (muS0, muR0)')
]
for xj, c, lab in jam_lines:
    ax_top.axvline(x=xj, color=c, linestyle='--', linewidth=1.5, label=lab)
    ax_bot.axvline(x=xj, color=c, linestyle='--', linewidth=1.5)

ax_top.text(-0.15, 0.95, '(a)', transform=ax_top.transAxes,
            fontsize=14, fontweight='bold', va='top')
ax_bot.text(-0.15, 0.95, '(b)', transform=ax_bot.transAxes,
            fontsize=14, fontweight='bold', va='top')

"""ax_top.legend(fontsize=9)
ax_bot.legend(fontsize=9)"""
plt.tight_layout()
plt.show()